### Ready to Go? Let's Start Your Chemical Discovery with IR-Psychoactive Substances-Classifiers!

💡 **Pro Tip:** Want to speed things up? You can use a powerful T4 GPU for free!

Just click on **Runtime** at the top-right of the page, select **Change runtime type**, and choose **T4 GPU** from the *Hardware accelerator* dropdown.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Final Pre-Flight Check (Crucial!)

1.  **Check Your Connection Status** at the top-right of the page.
    - Colab often connects automatically to your last-used device. If you don't see `Connecting...`, please click the **`Connect`** button manually.
2.  **Wait for the Green Checkmark** (✓) to appear next to your `RAM` and `Disk` status. This confirms you are **"Connected"**.
3.  When Colab asks for permission to access your files, go ahead and click **"Connect to Google Drive"** to authorize it.
4.  Once connected, Run Cells **"Step-by-Step"**.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### ✨ The PS-IR-Classifier tells you if a substance is psychoactive directly from its IR spectrum, no complex prep needed. After running, get ready to see:

* **1.Batch test results on our test set**
* **2.NPS test results**
* **3.Test a single file for PS classification**


---

### **Step 1: Project Setup & Data Download** ⚙️

<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0; font-size: 1.1em;">
        Your Automated Workspace Setup
    </h4>
    <p>
        This first code cell is the starting point for your entire discovery journey. It will automatically handle all the essential preparations for you.
    </p>
    <p>
        <strong>What it does:</strong>
    </p>
    <ul style="margin-top: 0; padding-left: 20px;">
        <li><strong>Connects to your Google Drive</strong> to save and access data.</li>
        <li><strong>Clones or updates our GitHub repository</strong> to ensure you're using the latest code.</li>
        <li><strong>Installs all required Python libraries</strong> to power the analysis.</li>
        <li><strong>Downloads all model weights and datasets</strong> from Hugging Face.</li>
    </ul>
    <p style="font-weight: bold;">
        To begin, simply run the code cell directly below this one.
    </p>
</div>

In [ ]:
#@title 🧠 1: Setting Up Your Workspace
# ==============================================================================
#  CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download # Import at the top

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Use force_remount for robustness
print("✅ Google Drive mounted.")


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================
print("\n--- Defining Project Paths ---")
# Project Root
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"

# Requirements file
REQUIREMENTS_FILE = os.path.join(GDRIVE_REPO_PATH, 'requirements_colab.txt')

# --- Data and Model Destination Folders on Google Drive ---
# These are the folders where downloaded files will be saved.
DEST_PS_CLASSIFIER_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS-Classifier', 'check_points')
print("✅ Paths defined.")


# ==================================================================
# --- 3. Clone or Update the 'CSU-IR' repository ---
# ==================================================================
print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your GitHub Personal Access Token in Colab Secrets (secret name: GITHUB_PAT)")

if not os.path.exists(GDRIVE_REPO_PATH):
    print(f"⏳ Cloning 'CSU-IR' to: {GDRIVE_REPO_PATH}")
    !git clone -q https://{GIT_TOKEN}@github.com/Hsqcsu/CSU-IR.git {GDRIVE_REPO_PATH}
else:
    print(f"✅ Repository already exists. Pulling latest changes...")
    !cd {GDRIVE_REPO_PATH} && git pull


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                shutil.copy(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully.")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"
PS_Classifier_MODEL_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/PS_Classifier/best_ir_model_650-4000.pth": "best_ir_model_650-4000.pth",
    "Model_weights/PS_Classifier/best_smiles_model_650-4000.pth": "best_smiles_model_650-4000.pth",
    "Model_weights/PS_Classifier/best_IR_Classifier_model.pth": "best_IR_Classifier_model.pth",
    "Model_weights/PS_Classifier/best_Smiles_Classifier_model.pth": "best_Smiles_Classifier_model.pth",
}
print("✅ Download helper and file lists are ready.")

# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")
download_files_from_hf(HF_REPO_ID, PS_Classifier_MODEL_WEIGHTS_TO_DOWNLOAD, DEST_PS_CLASSIFIER_WEIGHTS_PATH, HF_TOKEN)

# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")
print('--------------------------------------------------------------------------------------')
print("\nInstallation complete. The runtime will now restart automatically.")
print("Please wait for the session to reconnect and then run the next cell.")

# This command will automatically restart the Colab runtime.
import os
os.kill(os.getpid(), 9)
print("After re-connected, you can run the analysis and training notebooks.")

---
### **Step 2: Model Initialization & Data Loading**
<br>

<div style="background-color: #fffbe6; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #856404; margin-top: 0;">
        ⚠️ IMPORTANT: Please Read Before Running!
    </h4>
    <p>
        If the code cell below fails on the <strong>first run</strong> with a <code>NumPy 2.0</code> compatibility error, please do not worry. This is a known issue.
    </p>
    <p>
        <strong>Solution: Simply re-run the same cell a second time.</strong>
    </p>
    <p style="font-size: 0.9em; color: #666;">
        <em><strong>Technical Reason:</strong> You might see an error like <code>A module that was compiled using NumPy 1.x cannot be run in NumPy 2.0.x</code>. This happens because some pre-compiled packages (like rdkit) need the environment to adjust to the new NumPy version. The second run typically resolves this dependency conflict.</em>
    </p>
</div>

In [ ]:
#@title 🧠 2. Global Initialization (Load All Models & Data)
import sys
import os
import gzip
import torch
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from rdkit.Chem import Draw, MolFromSmiles
from google.colab import files
import pandas as pd
import jcamp
import numpy as np
from PIL import Image
import io

# --- 1. Set up project paths ---
print("--- Setting up project paths for module imports ---")
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"
PS_CLASSIFIER_MODULE_PATH = os.path.join(GDRIVE_REPO_PATH, 'PS-Classifier')
if PS_CLASSIFIER_MODULE_PATH not in sys.path: sys.path.insert(0, PS_CLASSIFIER_MODULE_PATH)
print("✅ Project paths set.")

# --- 2. Import custom modules ---
print("\n--- Importing custom modules ---")
from model.IR_encoder import IRModel
from model.SMILES_encoder import SmilesModel
from model.Classifier_model import classifymodel
from data_process.ir_process import preprocess_spectra_higer_500, preprocess_spectra_lower_500
from test_and_infer.infer import get_topK_result
from test_and_infer.infer_SMILES_Classifier import normalize_smiles
print("✅ Custom modules imported.")

# --- 3. Define all file paths ---
print("\n--- Defining all file paths ---")
PS_IR_ENCODER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_ir_model_650-4000.pth")
PS_IR_CLASSIFIER_PATH = os.path.join(DEST_PS_CLASSIFIER_WEIGHTS_PATH, "best_IR_Classifier_model.pth")
TEST_DATA_PATH = os.path.join(PS_CLASSIFIER_MODULE_PATH, "data", "test_data")
SINGLE_IR_TEST_DATA_PATH_NON_PS = os.path.join(PS_CLASSIFIER_MODULE_PATH, "data", "test_data",'(3-MERCAPTOPROPYL)TRIMETHOXYSILANE, 95%.CSV')
SINGLE_IR_TEST_DATA_PATH_PS = os.path.join(PS_CLASSIFIER_MODULE_PATH, "data", "test_data",'#773 - N-desethyletonitazene HCl (Lot# 0603955-33).JDX')
print("✅ File paths defined.")

# --- 4. Initialize all models and load weights ---
print("\n--- Initializing and loading all models ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# Classifier Models
# Note: The encoders for classifiers are separate instances
ir_classifier_encoder = IRModel()
ir_classifier_model = classifymodel(dim=1024, num_classes=2)


ir_classifier_encoder.load_state_dict(torch.load(PS_IR_ENCODER_PATH, map_location=device))
ir_classifier_model.load_state_dict(torch.load(PS_IR_CLASSIFIER_PATH, map_location=device))


ir_classifier_encoder.to(device).eval()
ir_classifier_model.to(device).eval()
print("✅ Classifier models loaded.")



print("\n\n🎉🎉🎉 Global initialization complete! System is ready! 🎉🎉🎉")

### **Step 3: Perfromence**

In [ ]:
#@title 📈 Task : Instant Spectra ID (IR Classification)
#@markdown Choose a test mode. You can either see the results on our pre-loaded test set or upload your own file to get an instant classification.
#@markdown ---
test_mode_ir = "test a single file for PS classification" #@param ["See batch test results on our test set", "See NPS test results","test a single file for PS classification"]

# --- Define a new dataset class for IR Classification ---
class IRDataset(Dataset):
    def __init__(self, ir, labels): self.ir, self.labels = ir, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ir[i], self.labels[i]

class NPSDataset(Dataset):
    def __init__(self, ir): self.ir = ir
    def __len__(self): return len(self.ir)
    def __getitem__(self, i): return self.ir[i],

if test_mode_ir == "See batch test results on our test set":
    print("--- Running batch test for IR Classifier ---")
    # Load data
    test_ir = torch.load(os.path.join(TEST_DATA_PATH, "IR_Classifier", "test_ir.pt"))
    test_label = torch.load(os.path.join(TEST_DATA_PATH, "IR_Classifier", "test_labels.pt"))
    test_loader = DataLoader(IRDataset(test_ir, test_label), batch_size=208, shuffle=False)

    # Run inference
    all_preds, all_labels = [], []
    with torch.no_grad():
        for ir_batch, label_batch in tqdm(test_loader, desc="IR Batch Test"):
            # Use the correct classifier encoder model
            ir_features = ir_classifier_encoder(ir_batch.to(device)[:, 150:])
            logits = ir_classifier_model(ir_features)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(label_batch.cpu().numpy())

    # Calculate and print confusion matrix
    label_0_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 0)
    label_0_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 0 and p == 1)
    label_1_pred_1 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 1)
    label_1_pred_0 = sum(1 for p, l in zip(all_preds, all_labels) if l == 1 and p == 0)
    print(f"\n--- Batch Test Results ---")
    print(f"Number of non-PS predicted as non-PS: {label_0_pred_0}")
    print(f"Number of non-PS predicted as PS: {label_0_pred_1}")
    print(f"Number of PS predicted as PS: {label_1_pred_1}")
    print(f"Number of PS predicted as non-PS: {label_1_pred_0}")

elif test_mode_ir == "See NPS test results":
    print("--- Running  test for NPS ---")
    # Load data
    test_ir = torch.load(os.path.join(TEST_DATA_PATH, "IR_Classifier", "NPS_ir.pt"))
    test_loader = DataLoader(NPSDataset(test_ir), batch_size=208, shuffle=False)
    predictions = []
    with torch.no_grad():
        for ir_batch in tqdm(test_loader, desc="NPS Test"):
            ir_features = ir_classifier_encoder(ir_batch.to(device)[:, 150:])
            logits = ir_classifier_model(ir_features)
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())

    print("\n--- NPS Test Results ---")
    for i, pred in enumerate(predictions):
        category = "Psychoactive Substance" if pred == 1 else "Non-Psychoactive Substance"
        print(f"Sample {i+1}: Predicted as -> {category}")


elif test_mode_ir == "test a single file for PS classification":
    print("\n--- Running single file classification ---")

    # Define the prediction function specifically for this cell
    def predict_ir_classification(filepath, spectrum_type="absorbance spectrum"):
        print(f"Processing file: {os.path.basename(filepath)}")
        if filepath.lower().endswith('.csv'):
            # Assuming CSV has no header
            df = pd.read_csv(filepath, header=None)
            wavenumbers, transmittances = df.iloc[:, 0].values, df.iloc[:, 1].values
        elif filepath.lower().endswith('.jdx'):
            data = jcamp.jcamp_readfile(filepath)
            wavenumbers, transmittances = np.array(data['x']), np.array(data['y'])
        else:
            return "Error: Unsupported file format. Please upload a .csv or .jdx file."

        if spectrum_type != "absorbance spectrum": transmittances /= 100.0

        try:
            if wavenumbers[0] > 500: ir_data = preprocess_spectra_higer_500(wavenumbers, transmittances)
            else: ir_data = preprocess_spectra_lower_500(filepath, wavenumbers, transmittances)
        except Exception as e:
            return f"Error during preprocessing: {e}"

        ir_tensor = torch.tensor(ir_data, dtype=torch.float32).unsqueeze(0).to(device)[:,150:]

        with torch.no_grad():
            # Use the correct classifier models
            ir_feature = ir_classifier_encoder(ir_tensor)
            logits = ir_classifier_model(ir_feature)

        pred = torch.argmax(logits, dim=1).item()
        return "✅ Result: Psychoactive Substance" if pred == 1 else "✅ Result: Non-Psychoactive Substance"


    PS_result = predict_ir_classification(SINGLE_IR_TEST_DATA_PATH_PS, spectrum_type="absorbance spectrum")
    print(f"The result of {os.path.basename(SINGLE_IR_TEST_DATA_PATH_PS)} is \n{PS_result}")

    Non_PS_result = predict_ir_classification(SINGLE_IR_TEST_DATA_PATH_NON_PS, spectrum_type="transmittance spectrum")
    print(f"The result of {os.path.basename(SINGLE_IR_TEST_DATA_PATH_NON_PS)} is \n{Non_PS_result}")
